In [5]:
# ============================================================
# 🔧 MASTER SETUP — Run this FIRST in every notebook
# ============================================================

# ── 1. Mount Google Drive ────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# ── 2. Install packages (only installs what's missing) ───────
import importlib, subprocess, sys

PACKAGES = {
    "sentence_transformers": "sentence-transformers",
    "rank_bm25":             "rank-bm25",
    "faiss":                 "faiss-cpu",
    "openai":                "openai",
    "langgraph":             "langgraph",
    "ragas":                 "ragas",
    "pypdf":                 "pypdf",
    "loguru":                "loguru",
    "tenacity":              "tenacity",
    "sklearn":               "scikit-learn",
}

missing = []
for module, pkg in PACKAGES.items():
    try:
        importlib.import_module(module)
    except ImportError:
        missing.append(pkg)

if missing:
    print(f"⏳ Installing {len(missing)} missing packages: {missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)
    print("✅ Packages installed")
else:
    print("✅ All packages already installed")

# ── 3. Set all environment variables ─────────────────────────
import os
from google.colab import userdata

def get_secret(name, fallback=""):
    try:
        return userdata.get(name)
    except:
        return fallback

os.environ["OPENAI_API_KEY"]               = get_secret("OPENAI_API_KEY")
os.environ["OPENAI_MODEL"]                 = "gpt-4o-mini"
os.environ["EMBEDDING_MODEL"]              = "BAAI/bge-base-en-v1.5"
os.environ["EMBEDDING_DIMENSION"]          = "768"
os.environ["RERANKER_MODEL"]               = "cross-encoder/ms-marco-MiniLM-L-6-v2"
os.environ["BM25_WEIGHT"]                  = "0.3"
os.environ["VECTOR_WEIGHT"]                = "0.7"
os.environ["TOP_K_RETRIEVE"]               = "20"
os.environ["TOP_K_RERANK"]                 = "5"
os.environ["RETRIEVAL_QUALITY_THRESHOLD"]  = "0.5"
os.environ["MIN_FEEDBACK_SAMPLES"]         = "5"
os.environ["LEARNING_RATE"]               = "0.01"

# ── 4. Set all paths ─────────────────────────────────────────
PROJECT_DIR   = "/content/drive/MyDrive/SelfImprovingRAG"
RAW_DIR       = f"{PROJECT_DIR}/data/raw"
PROCESSED_DIR = f"{PROJECT_DIR}/data/processed"
FEEDBACK_DB   = f"{PROJECT_DIR}/feedback.db"

for d in [RAW_DIR, PROCESSED_DIR]:
    os.makedirs(d, exist_ok=True)

# ── 5. Load all saved artifacts (chunks + indexes) ───────────
import numpy as np, json, pickle, re, hashlib, math
from dataclasses import dataclass, field
from typing import List, Optional, Dict, Any
from pathlib import Path

# Data classes — must be defined before unpickling
@dataclass
class Chunk:
    chunk_id: str; doc_id: str; content: str
    metadata: Dict[str, Any] = field(default_factory=dict)
    parent_chunk_id: Optional[str] = None
    chunk_index: int = 0; total_chunks: int = 0
    def _generate_id(self):
        h = hashlib.md5(f"{self.doc_id}:{self.chunk_index}:{self.content[:100]}".encode()).hexdigest()
        return f"chunk_{h[:12]}"

@dataclass
class Document:
    doc_id: str; content: str
    metadata: Dict[str, Any] = field(default_factory=dict)
    source: str = ""

# Load chunks
all_parent_chunks, all_child_chunks = [], []
parent_pkl = f"{PROCESSED_DIR}/parent_chunks.pkl"
child_pkl  = f"{PROCESSED_DIR}/child_chunks.pkl"

if os.path.exists(parent_pkl) and os.path.exists(child_pkl):
    with open(parent_pkl, "rb") as f: all_parent_chunks = pickle.load(f)
    with open(child_pkl,  "rb") as f: all_child_chunks  = pickle.load(f)
    print(f"✅ Chunks loaded  →  {len(all_parent_chunks)} parents | {len(all_child_chunks)} children")
else:
    print("⚠️  No chunks found — run Notebook 2 first OR the self-healing cell in Notebook 3")

# Load embeddings
all_embeddings, child_ids = None, []
emb_path = f"{PROCESSED_DIR}/child_embeddings.npy"
ids_path  = f"{PROCESSED_DIR}/child_ids.json"

if os.path.exists(emb_path) and os.path.exists(ids_path):
    all_embeddings = np.load(emb_path)
    with open(ids_path) as f: child_ids = json.load(f)
    print(f"✅ Embeddings loaded  →  shape {all_embeddings.shape}")
else:
    print("⚠️  No embeddings found — run Notebook 3 first")

# Load BM25
bm25, bm25_ids = None, []
bm25_path = f"{PROCESSED_DIR}/bm25_index.pkl"

if os.path.exists(bm25_path):
    with open(bm25_path, "rb") as f: bm25_data = pickle.load(f)
    bm25     = bm25_data["bm25"]
    bm25_ids = bm25_data["chunk_ids"]
    print(f"✅ BM25 loaded  →  {len(bm25_ids)} docs")
else:
    print("⚠️  No BM25 index found — run Notebook 3 first")

# Load FAISS (rebuild in-memory from saved embeddings — always fast)
faiss_index, chunk_map = None, {}
faiss_bin = f"{PROCESSED_DIR}/faiss_index.bin"

if all_embeddings is not None:
    import faiss
    if os.path.exists(faiss_bin):
        faiss_index = faiss.read_index(faiss_bin)
    else:
        dim = all_embeddings.shape[1]
        faiss_index = faiss.IndexFlatIP(dim)
        emb_copy = all_embeddings.copy().astype(np.float32)
        faiss.normalize_L2(emb_copy)
        faiss_index.add(emb_copy)
    chunk_map = {c.chunk_id: c for c in all_child_chunks}
    print(f"✅ FAISS ready  →  {faiss_index.ntotal} vectors")

# ── 6. Print final status ─────────────────────────────────────
print("\n" + "="*45)
key_set = bool(os.environ.get("OPENAI_API_KEY","").startswith("sk-"))
print(f"{'✅' if key_set else '❌'} OpenAI Key : {'set' if key_set else 'MISSING — add to Colab Secrets'}")
print(f"✅ Project  : {PROJECT_DIR}")
print(f"✅ Model    : {os.environ['EMBEDDING_MODEL']}")
print(f"✅ Chunks   : {len(all_child_chunks)} child | {len(all_parent_chunks)} parent")
print(f"✅ FAISS    : {faiss_index.ntotal if faiss_index else 0} vectors")
print(f"✅ BM25     : {len(bm25_ids)} docs")
print("="*45)
print("🚀 Environment ready — continue with the next cell")

Mounted at /content/drive
⏳ Installing 5 missing packages: ['rank-bm25', 'faiss-cpu', 'ragas', 'pypdf', 'loguru']
✅ Packages installed
✅ Chunks loaded  →  1 parents | 5 children
✅ Embeddings loaded  →  shape (5, 768)
✅ BM25 loaded  →  5 docs
✅ FAISS ready  →  5 vectors

✅ OpenAI Key : set
✅ Project  : /content/drive/MyDrive/SelfImprovingRAG
✅ Model    : BAAI/bge-base-en-v1.5
✅ Chunks   : 5 child | 1 parent
✅ FAISS    : 5 vectors
✅ BM25     : 5 docs
🚀 Environment ready — continue with the next cell


In [6]:
# Run env CONFIG first, then:
import numpy as np, json, pickle, os, faiss, re

PROCESSED_DIR = "/content/drive/MyDrive/SelfImprovingRAG/data/processed"

# Load BM25
with open(f"{PROCESSED_DIR}/bm25_index.pkl", "rb") as f:
    bm25_data = pickle.load(f)
bm25 = bm25_data["bm25"]
bm25_ids = bm25_data["chunk_ids"]

# Load chunks
with open(f"{PROCESSED_DIR}/child_chunks.pkl", "rb") as f:
    all_child_chunks = pickle.load(f)

chunk_map = {c.chunk_id: c for c in all_child_chunks}

from sentence_transformers import SentenceTransformer
embed_model = SentenceTransformer(os.environ.get("EMBEDDING_MODEL", "BAAI/bge-base-en-v1.5"))

print("✅ All indexes loaded")
print(f"   FAISS: {faiss_index.ntotal} vectors")
print(f"   BM25: {len(bm25_ids)} documents")
print(f"   Chunks: {len(chunk_map)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ All indexes loaded
   FAISS: 5 vectors
   BM25: 5 documents
   Chunks: 5


In [7]:
from dataclasses import dataclass
from typing import List, Dict, Optional, Tuple

STOPWORDS = {"the","a","an","and","or","but","in","on","at","to","for","of","with",
             "is","was","are","were","be","been","have","has","had","do","does","did",
             "will","would","could","should","may","might","this","that","it"}

def tokenize(text):
    text = re.sub(r'[^\w\s-]', ' ', text.lower())
    return [t for t in text.split() if t not in STOPWORDS and len(t) > 1]

@dataclass
class SearchResult:
    chunk_id: str
    content: str
    score: float
    bm25_rank: int = 0
    vector_rank: int = 0
    rrf_score: float = 0.0
    metadata: Dict = None
    parent_chunk_id: Optional[str] = None

    def __post_init__(self):
        if self.metadata is None:
            self.metadata = {}

def hybrid_search(query: str, top_k: int = 10, bm25_w: float = 0.3, vec_w: float = 0.7):
    """Full hybrid search with RRF fusion."""
    K = 60  # RRF smoothing constant

    # 1. BM25 search
    tokens = tokenize(query)
    bm25_scores = bm25.get_scores(tokens)
    bm25_top = bm25_scores.argsort()[::-1][:top_k * 3]
    bm25_ranked = [(bm25_ids[i], float(bm25_scores[i])) for i in bm25_top if bm25_scores[i] > 0]

    # 2. Vector search (FAISS)
    prefix = "Represent this sentence for searching relevant passages: "
    q_emb = embed_model.encode([prefix + query], normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)
    D, I = faiss_index.search(q_emb, k=top_k * 3)
    vec_ranked = [(bm25_ids[idx], float(D[0][rank])) for rank, idx in enumerate(I[0]) if idx >= 0]

    # 3. RRF Fusion
    rrf_scores = {}
    for rank, (cid, _) in enumerate(bm25_ranked):
        rrf_scores[cid] = rrf_scores.get(cid, 0) + bm25_w / (K + rank + 1)
    for rank, (cid, _) in enumerate(vec_ranked):
        rrf_scores[cid] = rrf_scores.get(cid, 0) + vec_w / (K + rank + 1)

    sorted_ids = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

    results = []
    for rank, (cid, rrf_score) in enumerate(sorted_ids):
        if cid in chunk_map:
            chunk = chunk_map[cid]
            bm25_rank_pos = next((i for i, (id_, _) in enumerate(bm25_ranked) if id_ == cid), 999)
            vec_rank_pos = next((i for i, (id_, _) in enumerate(vec_ranked) if id_ == cid), 999)
            results.append(SearchResult(
                chunk_id=cid, content=chunk.content,
                score=rrf_score, rrf_score=rrf_score,
                bm25_rank=bm25_rank_pos, vector_rank=vec_rank_pos,
                metadata=chunk.metadata, parent_chunk_id=chunk.parent_chunk_id
            ))
    return results

print("✅ HybridSearcher ready")

✅ HybridSearcher ready


In [8]:
# Test with a real query about your documents
test_queries = [
    "What is retrieval augmented generation?",
    "How does attention mechanism work?",
    "What are the limitations of large language models?"
]

for query in test_queries:
    print(f"\n{'='*60}")
    print(f"🔍 Query: {query}")
    print(f"{'='*60}")
    results = hybrid_search(query, top_k=5)

    for i, r in enumerate(results):
        print(f"\nRank {i+1}:")
        print(f"  RRF Score: {r.rrf_score:.5f}")
        print(f"  BM25 Rank: {r.bm25_rank+1 if r.bm25_rank < 999 else 'Not found'}")
        print(f"  Vector Rank: {r.vector_rank+1 if r.vector_rank < 999 else 'Not found'}")
        print(f"  Content: {r.content[:150]}...")


🔍 Query: What is retrieval augmented generation?

Rank 1:
  RRF Score: 0.01148
  BM25 Rank: Not found
  Vector Rank: 1
  Content: Retrieval Augmented Generation (RAG) enhances LLMs with external knowledge. Retrieval Augmented Generation (RAG) enhances LLMs with external knowledge...

Rank 2:
  RRF Score: 0.01129
  BM25 Rank: Not found
  Vector Rank: 2
  Content: Retrieval Augmented Generation (RAG) enhances LLMs with external knowledge. Retrieval Augmented Generation (RAG) enhances LLMs with external knowledge...

Rank 3:
  RRF Score: 0.01111
  BM25 Rank: Not found
  Vector Rank: 3
  Content: Retrieval Augmented Generation (RAG) enhances LLMs with external knowledge. Retrieval Augmented Generation (RAG) enhances LLMs with external knowledge...

Rank 4:
  RRF Score: 0.01094
  BM25 Rank: Not found
  Vector Rank: 4
  Content: Retrieval Augmented Generation (RAG) enhances LLMs with external knowledge. Retrieval Augmented Generation (RAG) enhances LLMs with external knowledge...

Rank 5:
 

In [9]:
import pandas as pd

def bm25_only_search(query, top_k=5):
    tokens = tokenize(query)
    scores = bm25.get_scores(tokens)
    top = scores.argsort()[::-1][:top_k]
    return [(bm25_ids[i], float(scores[i]), all_child_chunks[i].content[:100]) for i in top if scores[i] > 0]

def vector_only_search(query, top_k=5):
    prefix = "Represent this sentence for searching relevant passages: "
    q_emb = embed_model.encode([prefix + query], normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)
    D, I = faiss_index.search(q_emb, k=top_k)
    return [(bm25_ids[I[0][i]], float(D[0][i]), all_child_chunks[I[0][i]].content[:100]) for i in range(len(I[0])) if I[0][i] >= 0]

query = "What is retrieval augmented generation?"
print(f"Query: {query}\n")
print("BM25 Only:")
for cid, score, content in bm25_only_search(query):
    print(f"  {score:.3f} | {content}...")

print("\nVector Only:")
for cid, score, content in vector_only_search(query):
    print(f"  {score:.4f} | {content}...")

print("\nHybrid (RRF):")
for r in hybrid_search(query):
    print(f"  {r.rrf_score:.5f} | BM25:{r.bm25_rank+1} Vec:{r.vector_rank+1} | {r.content[:100]}...")

Query: What is retrieval augmented generation?

BM25 Only:

Vector Only:
  0.8441 | Retrieval Augmented Generation (RAG) enhances LLMs with external knowledge. Retrieval Augmented Gene...
  0.8434 | Retrieval Augmented Generation (RAG) enhances LLMs with external knowledge. Retrieval Augmented Gene...
  0.8404 | Retrieval Augmented Generation (RAG) enhances LLMs with external knowledge. Retrieval Augmented Gene...
  0.8377 | Retrieval Augmented Generation (RAG) enhances LLMs with external knowledge. Retrieval Augmented Gene...
  0.8336 | Retrieval Augmented Generation (RAG) enhances LLMs with external knowledge. Retrieval Augmented Gene...

Hybrid (RRF):
  0.01148 | BM25:1000 Vec:1 | Retrieval Augmented Generation (RAG) enhances LLMs with external knowledge. Retrieval Augmented Gene...
  0.01129 | BM25:1000 Vec:2 | Retrieval Augmented Generation (RAG) enhances LLMs with external knowledge. Retrieval Augmented Gene...
  0.01111 | BM25:1000 Vec:3 | Retrieval Augmented Generation (RAG) en